# TAP access to CSA metadata

To install [astroquery](https://astroquery.readthedocs.io/en/latest/index.html) ([Ginsburg et al, 2019](https://doi.org/10.3847/1538-3881/aafc33)) to be able to use [TAP+](https://astroquery.readthedocs.io/en/latest/utils/tap.html):

`conda install -c astropy astroquery`

The different missions have their own TAP servers; Cluster's is:
`https://csa.esac.esa.int/csa-sl-tap/tap/`

## Tables: metadata of metadata

First of all, it's helpful to know what tables are there. 

From example 1.1 on [Astroquery TAP page](https://astroquery.readthedocs.io/en/latest/utils/tap.html), load the table objects using the method `load_tables()`, then print the names of those table objects using the `get_qualified_name()` method. This returns the schema (e.g., csa.) and the table names (e.g., csa.dataset). We only need to focus on the `csa.` schema tables.

In [1]:
from astroquery.utils.tap.core import TapPlus

CSA = TapPlus(url="https://csa.esac.esa.int/csa-sl-tap/tap/")
tables = CSA.load_tables()#only_names=True)

for table in tables:
    print(table.get_qualified_name())

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]
csa.array_dimension
csa.cluster_data_type
csa.configuration
csa.dataset
csa.dataset_contact
csa.dataset_inventory
csa.dataset_referenced
csa.experiment
csa.experiment_dataset
csa.file
csa.instrument_type
csa.measurement_type
csa.news
csa.observatory
csa.parameter
csa.parameter_data
csa.parameter_dependency
csa.parameter_label
csa.parameter_representation
csa.pregen_inventory
csa.processing_level
csa.v_dataset
csa.v_dataset_inventory
csa.v_dataset_referenced
csa.v_experiment
csa.v_file
csa.v_instrument
csa.v_mission
csa.v_observatory
csa.v_parameter
csa.v_pregen_inventory
csa.v_quick_look_dataset
graphics.distribution_panel
graphics.panel
graphics.pregen_file
public.dual
tap_config.coord_sys
tap_config.properties
tap_schema.columns
tap_schema.key_columns
tap_schema.keys
tap_schema.schemas
tap_schema.tables


From the above list of tables, the helpful ones are those beginning `csa.csa.` and the ones meant for users begin with `csa.csa.v_`, where the 'v' means 'view'. The view tables contain the same information as the other `csa.csa.` tables but the columns in those tables might be grouped slightly differently, and the columns containing information meant for internal database use have been taken out since they are of no use to the end-user.

The schema (the first `csa.`) isn't needed when making the query.

Also from [Astroquery TAP page](https://astroquery.readthedocs.io/en/latest/utils/tap.html), load a specific table and print out the column names. 

In [2]:
table = CSA.load_table('csa.v_parameter')

print('\n Table: \n', table, '\n')

print('Column names: ')
for column in (table.columns):
    print(column.name)


 Table: 
 TAP Table name: csa.v_parameter
Description: None
Size (bytes): 0
Num. columns: 37 

Column names: 
array_dimension
cat_description
caveat
compound
compound_definition
coordinate_system
dataset_id
data_type
delta_minus
delta_plus
display_type
entity
error_minus
error_plus
field_name
fillval
fluctuations
frame
frame_origin
frame_velocity
label_axis
parameter_data
parameter_dependency
parameter_id
parameter_label
parameter_representation
property
quality
scale_max
scale_min
scale_type
si_conversion
significant_digits
target_system
tensor_order
units
value_type


## Querying Tables - accessing metadata

So far we have only been loading metadata of the metadata, so we need to make an actual query to get the values:

From [AppDynamics](https://docs.appdynamics.com/display/PRO45/ADQL+Queries) rather than [Astronomical Data Query Language](https://www.ivoa.net/documents/ADQL/2.0), but still usefully put:

`SELECT * | {[DISTINCT] expression [AS alias] [, expression [AS alias]]...}
FROM event_type
[WHERE condition_expression]
[SINCE timevalue [UNTIL timevalue]]
[HAVING condition_expression]
[ORDER BY {field_name | alias} [ASC | DESC] [, {field_name | alias} [ASC | DESC]]...]
[LIMIT integer [, integer]...]`

While the SELECT and FROM clauses are required, all other clauses are optional. Type spaces around each keyword. 

### Whole table

To see all the columns and first 2000 rows (TAP+ has a synchronous request limit of 2000) of a table:

In [3]:
results = CSA.launch_job("SELECT * FROM csa.v_dataset")
r = results.get_results()
r

acknowledgement,allow_overlap,category,compress,crosscal,dataset_caveats,dataset_id,dataset_reference,dataset_version,data_type,data_type_id,date_last_file_ingested,date_last_update,description,display_order,end_date,experiments,file_extension,gui_name,ingestion_date,instruments,interval,inventory,is_cef,is_istp,key_product,level,main_group,max_start_file_date,max_time_resolution,measurement_types,min_time_resolution,multi_inst,multi_sc,observatory_name,on_demand,person_reference,pregen_product,quicklook_product,released,start_date,time_resolution,title,unit_name
str2000,int16,str50,int16,int16,str3000,object,object,str80,str100,str2,object,object,str4500,int32,object,object,str10,object,object,object,str60,int16,bool,bool,int16,str50,str20,object,float64,object,float64,int16,int16,str20,int16,object,int16,int16,int16,object,float64,object,str50
Please acknowledge the instrument team and ESA Cluster Science Archive in any publication based upon use of this data. The DWP instrument DOI is https://doi.org/10.5270/esa-ftdfdba.,0,General,1,0,See User Guide for general caveat information. *C2_CQ_DWP_PCOR_FX This caveat list is complemented by the PEACE operational log. www.mssl.ucl.ac.uk/missions/cluster/about_operations/PEACE_ops_history.php,C2_CP_DWP_PCOR_FX,"C2_CQ_DWP_INST (1), C2_CQ_DWP_PCOR_FX (1)",,CAA_Parameter,CP,2026-02-12T21:21:09.697Z,2026-07-06T13:47:16.840Z,"The DWP particle correlator data sets provide the result of the onboard autocorrelation of the PEACE HEEA count rate. Due to telemetry restrictions, only two of the possible 16 correlator energy channels are returned. One data set, PCOR_FX (this file) provides ACFs at a fixed energy whilst the other PCOR_ST data set steps through the remaining energy steps at a rate of one step per spin. The 32 ACF lag values provide count rate information in the range 1.4-41.6kHz at a particular energy. In normal mode, one ACF is produced every spin where as in burst modes there may be either 4, 8, or 16 ACFs per spin depending upon the WEC operational mode implemented. In addition to the ACF and energy, this data set also contains the HEEA polar sensor from which the data were collected, the azimuth of observations (BM only), the maximum and minimum pitch angles of the electrons recorded and an estimate of the count rate based on the ACF. For further details, the user is referred to the DWP User Manual, available from CSA, which contains a complete description of all parameters within the data set as well as a list of the general caveats that apply to this data set.",10,2024-09-30T00:00:00.000Z,DWP,cef,"Particle Correlator High Resolution Data, fixed energy band",2021-02-18T12:02:20.201Z,DWP2,,1,True,False,0,Derived,Science,2024-09-30T00:00:00.000Z,0.22,Status,4.4,0,0,Cluster-2,0,K Yearby>Instrument Manager>K.H.Yearby@sheffield.ac.uk ---- S Walker>Data Manager>Simon.Walker@sheffield.ac.uk ---- A M Buckley>DWP correlator scientist>A.M.Buckley@sussex.ac.uk,0,0,1,2000-07-16T00:00:00.000Z,4.0,"DWP Particle Correlator High Resolution Data, fixed energy band",
Please acknowledge the ESA Cluster Active Archive in any publication based upon use of this data,0,General,1,0,See User Guide for caveats,C3_CG_AUX_CAA_6HR_OVERVIEW_PS,,,CAA_Graphic,CG,2026-08-20T16:47:36.827Z,2026-07-17T02:12:11.907Z,"This plot, available for each spacecraft, is based on the calibrated CAA data and has a similar layout as the corresponding CSDSweb plot (that is based on default calibration values). The panels from top to bottom are: Total magnetic field (FGM) Spacecraft potential (EFW) Total ion velocity (CIS) Ion energy spectra (CIS) Electron energy spectra (PEACE) Magnetic field wave spectra at 10 Hz - 4 kHz (STAFF) Electric field wave spectra at 2-80 kHz (WHISPER)",30100,2024-10-01T00:00:00.000Z,AUX,ps,6-hr CAA Summary Plot (Overview),2015-03-03T16:41:47.471Z,AUX3,,0,False,False,0,Calibrated,Graphical,2024-09-30T18:00:00.000Z,21600.0,Status,21600.0,0,0,Cluster-3,0,caaops>Non-PI>caaops@sciops.esa.int,0,0,1,200

### Using WHERE to restrict results

To get a list of all datasets that include FGM, we need to query the v_dataset table for the dataset_id and add the WHERE statement to restrict the results to those that contain 'FGM'. Remember that the percentage sign (%) is the wildcard, but it doesn't need to be URL encoded here.

In [4]:
results = CSA.launch_job("SELECT dataset_id FROM csa.v_dataset WHERE dataset_id like '%FGM%'")
r = results.get_results()
r

dataset_id
object
CM_CD_FGM_CR_ICD_UG
C4_PP_FGM
CL_SP_FGM
C4_CQ_FGM_CAVF
C4_CQ_FGM_GAPF
C4_CQ_FGM_VALF
C4_CP_FGM_5VPS_ISR2
C4_CP_FGM_5VPS
C4_CP_FGM_FULL_ISR2


### An inventory file request:

Request (SELECT) 5 columns FROM csa.v_dataset_inventory table WHERE the dataset_id is 'C1_CP_FGM_SPIN' AND start_time is *before* 1 May 2002 AND end_time is *after* 1 April 2002 and ORDER BY the start time column, and print the results, as text or as an [astropy.table](https://docs.astropy.org/en/stable/table/index.html?highlight=astropy.table):

In [5]:
results = CSA.launch_job("SELECT dataset_id,start_time,end_time,num_instances,inventory_version "
                         "FROM csa.v_dataset_inventory "
                         "WHERE dataset_id='C1_CP_FGM_SPIN' "
                         "AND start_time<='2002-05-01T00:00:00' " 
                         "AND end_time>='2002-04-01T00:00:00' "
                         "ORDER BY start_time")

# Either print the results as text, or as an astropy.table (which can be easier to read)
#print(results.get_results())

r = results.get_results()
r

dataset_id,start_time,end_time,num_instances,inventory_version
object,object,object,int32,int32
C1_CP_FGM_SPIN,2002-03-30T18:17:57.879Z,2002-04-01T00:00:00.000Z,0,10
C1_CP_FGM_SPIN,2002-04-01T00:00:00.000Z,2002-04-01T00:21:04.972Z,0,10
C1_CP_FGM_SPIN,2002-04-01T00:21:04.972Z,2002-04-01T00:28:51.002Z,117,10
C1_CP_FGM_SPIN,2002-04-01T00:28:51.002Z,2002-04-01T00:35:16.683Z,0,10
C1_CP_FGM_SPIN,2002-04-01T00:35:16.683Z,2002-04-01T03:36:48.128Z,2704,10
C1_CP_FGM_SPIN,2002-04-01T03:36:48.128Z,2002-04-01T18:41:24.702Z,0,10
C1_CP_FGM_SPIN,2002-04-01T18:41:24.702Z,2002-04-01T18:48:50.646Z,112,10
C1_CP_FGM_SPIN,2002-04-01T18:48:50.646Z,2002-04-01T18:55:20.346Z,0,10
C1_CP_FGM_SPIN,2002-04-01T18:55:20.346Z,2002-04-02T06:44:49.979Z,10534,10
